# JPSS VIIRS True Color with a user-defined domain

## Goal

Load VIIRS SDR spectral and geolocation files, enter a geographic domain in decimal longitude/latitude, create a True Color composite, save it as a PNG, and display the image inside JupyterLab.

The example domain around Shishaldin is `(-166.0, 54.0, -162.0, 56.0)`. It is editable input, not an automatic preset.

### Reference image

The image below is a reference NOAA-21 VIIRS global True Color composite. The image generated from your swath appears near the end of the notebook.

![Reference NOAA-21 VIIRS True Color imagery](https://www.star.nesdis.noaa.gov/data/smcd1/icvs_metrics/status/J02/VIIRS/2026/07/20260723_J02_VIIRS_Global_IMAGE_RGB543.jpg)

Reference imagery: [NOAA STAR VIIRS instrument status](https://www.star.nesdis.noaa.gov/icvs/status_N21_VIIRS.php).

## Setup

Run the next cell once in a new environment. Restart the kernel after installation if Jupyter asks you to.

In [ ]:
%pip install -r ../requirements-notebooks.txt

In [ ]:
from pathlib import Path
import sys

from IPython.display import Image, display
from PIL import Image as PillowImage
from satpy import Scene

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from examples.render_satellite import (
    crop_and_resample_scene,
    expand_inputs,
    has_viirs_geolocation,
    validate_bbox,
)

## Steps

### 1. Enter the files, RGB, and domain

Write all four domain values as decimal degrees. The order is `MIN_LON, MIN_LAT, MAX_LON, MAX_LAT`.

In [ ]:
FILES_GLOB = str(REPO_ROOT / "data" / "viirs" / "*.h5")
COMPOSITE = "true_color"

# User-defined Shishaldin example in decimal degrees.
DOMAIN = (-166.0, 54.0, -162.0, 56.0)

OUTPUT = REPO_ROOT / "output" / "viirs_shishaldin_true_color.png"

validate_bbox(DOMAIN)

### 2. Find and check the VIIRS files

True Color normally uses `M03`, `M04`, and `M05`; `I01` and `I02` support ratio sharpening. Keep the matching `GITCO` and `GMTCO` terrain-corrected geolocation files from the same granule in the directory.

In [ ]:
files = expand_inputs([FILES_GLOB])
if not files:
    raise FileNotFoundError(
        f"No VIIRS HDF5 files matched {FILES_GLOB}. "
        "Download matching spectral and geolocation files first."
    )

if not has_viirs_geolocation(files):
    raise FileNotFoundError(
        "No VIIRS geolocation file was found. Add the matching GITCO/GMTCO files."
    )

print(f"Found {len(files)} VIIRS files")
for filename in files[:12]:
    print(Path(filename).name)

### 3. Load the RGB composite

In [ ]:
scene = Scene(reader="viirs_sdr", filenames=files)
available = {str(name) for name in scene.available_composite_names()}

if COMPOSITE not in available:
    raise ValueError(
        f"{COMPOSITE!r} is unavailable. Check that the bands and geolocation "
        "files belong to the same granule."
    )

scene.load([COMPOSITE], generate=True)
print(f"Loaded composite: {COMPOSITE}")

### 4. Crop, resample, and save

The requested domain must intersect the downloaded VIIRS swath.

In [ ]:
output_scene = crop_and_resample_scene(scene, domain=DOMAIN)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
output_scene.save_dataset(
    COMPOSITE,
    filename=str(OUTPUT),
    writer="simple_image",
)
print(f"Image created: {OUTPUT.resolve()}")

### 5. Display the generated satellite image

In [ ]:
display(Image(filename=str(OUTPUT)))

## Checks

Confirm that the output exists, has nonzero dimensions, contains the requested region, and shows no displacement between I and M bands.

In [ ]:
if not OUTPUT.exists():
    raise FileNotFoundError(OUTPUT)

with PillowImage.open(OUTPUT) as image:
    print(f"PNG size: {image.width} x {image.height} pixels")
    assert image.width > 0 and image.height > 0

## Next steps

- Change all four decimal `DOMAIN` values for another study area.
- Confirm pass coverage before downloading large VIIRS granules.
- Try another available composite after downloading its required bands.
- Keep spectral and geolocation files from the same satellite, date, time, and granule.